In [0]:
import logging, json, time
from datetime import datetime, timezone, timedelta

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("riskbricks.ml_predictions")

def log_step(step_name, table_name=None, row_count=None, error=None):
    entry = {"step": step_name, "status": "ERROR" if error else "OK", "timestamp": datetime.now(timezone.utc).isoformat()}
    if table_name: entry["table"] = table_name
    if row_count is not None: entry["rows"] = row_count
    if error: entry["error"] = str(error)[:300]
    logger.info(json.dumps(entry))

dbutils.widgets.text('catalog', 'riskbricks')
CATALOG = dbutils.widgets.get('catalog').strip()

from pyspark.sql import functions as F, Window
import numpy as np
import pandas as pd

FOCUS_SYMBOLS = [
    "LMT", "RTX", "NOC", "GD", "BA", "HII",
    "XOM", "CVX", "COP", "SLB", "HAL", "OXY",
    "JPM", "BAC", "GS", "MS", "C", "WFC",
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA",
    "INTC", "AMD", "AVGO", "QCOM", "MU", "LRCX", "AMAT",
    "WMT", "COST", "HD", "NKE", "MCD", "SBUX",
    "JNJ", "PFE", "UNH", "LLY", "ABBV", "MRK",
    "CAT", "DE", "HON", "GE", "MMM",
    "UAL", "DAL", "AAL",
]
sym_list = ", ".join(f"'{s}'" for s in FOCUS_SYMBOLS)

CURATED_FEATURES = [
    "return_5d", "return_20d", "volatility_20d",
    "ai_sentiment", "news_count",
    "gdelt_tone", "gdelt_events",
    "rsi_14", "macd_hist", "gap_pct",
    "sector_momentum_5d", "sector_breadth",
    "advance_ratio", "pct_above_ma20",
    "vix", "days_to_earnings", "is_monday",
]

spark.sql(f'USE CATALOG {CATALOG}')
print(f'\u2705 Config ready | Catalog: {CATALOG} | Symbols: {len(FOCUS_SYMBOLS)} | Features: {len(CURATED_FEATURES)}')

In [0]:
# ── Step 1: Assemble gold.ml_prediction_features ────────────────────────────
# Joins all silver/bronze sources into the 17-feature prediction vector
try:
    features = spark.sql(f"""
        WITH prices AS (
            SELECT symbol, last_close, return_5d, return_20d, volatility_20d, as_of_date
            FROM {CATALOG}.silver.forecast_features_daily
            WHERE symbol IN ({sym_list})
              AND as_of_date = (SELECT MAX(as_of_date) FROM {CATALOG}.silver.forecast_features_daily)
        ),
        tech AS (
            SELECT * FROM {CATALOG}.silver.technical_indicators
            WHERE date = (SELECT MAX(date) FROM {CATALOG}.silver.technical_indicators WHERE symbol IN ({sym_list}))
        ),
        sector AS (
            SELECT * FROM {CATALOG}.silver.sector_features
            WHERE date = (SELECT MAX(date) FROM {CATALOG}.silver.sector_features)
        ),
        breadth AS (
            SELECT * FROM {CATALOG}.silver.market_breadth
            WHERE date = (SELECT MAX(date) FROM {CATALOG}.silver.market_breadth)
        ),
        macro AS (
            SELECT indicator, value FROM {CATALOG}.bronze.fred_macro_indicators
            WHERE date = (SELECT MAX(date) FROM {CATALOG}.bronze.fred_macro_indicators WHERE indicator = 'VIX')
        ),
        ai AS (SELECT * FROM {CATALOG}.silver.news_ai_sentiment),
        gdelt AS (
            SELECT symbol, AVG(avg_tone) AS gdelt_tone, COUNT(*) AS gdelt_events
            FROM {CATALOG}.bronze.historical_news_gdelt
            WHERE event_date >= DATE_SUB(CURRENT_DATE(), 5) AND symbol IN ({sym_list})
            GROUP BY symbol
        )
        SELECT p.symbol, p.as_of_date AS pred_date, p.last_close,
               p.return_5d, p.return_20d, p.volatility_20d,
               COALESCE(a.ai_sentiment, 0) AS ai_sentiment,
               COALESCE(a.news_count, 0) AS news_count,
               COALESCE(g.gdelt_tone, 0) AS gdelt_tone,
               COALESCE(g.gdelt_events, 0) AS gdelt_events,
               COALESCE(t.rsi_14, 50) AS rsi_14,
               COALESCE(t.macd_hist, 0) AS macd_hist,
               COALESCE(t.gap_pct, 0) AS gap_pct,
               COALESCE(s.sector_momentum_5d, 0) AS sector_momentum_5d,
               COALESCE(s.sector_breadth, 0.5) AS sector_breadth,
               COALESCE(b.advance_ratio, 0.5) AS advance_ratio,
               COALESCE(b.pct_above_ma20, 0.5) AS pct_above_ma20,
               COALESCE(MAX(CASE WHEN m.indicator='VIX' THEN m.value END), 20) AS vix,
               30 AS days_to_earnings,
               CASE WHEN DAYOFWEEK(p.as_of_date) = 2 THEN 1 ELSE 0 END AS is_monday
        FROM prices p
        LEFT JOIN ai a ON p.symbol = a.symbol
        LEFT JOIN gdelt g ON p.symbol = g.symbol
        LEFT JOIN tech t ON p.symbol = t.symbol
        LEFT JOIN sector s ON p.symbol = s.symbol
        CROSS JOIN breadth b
        CROSS JOIN macro m
        GROUP BY p.symbol, p.as_of_date, p.last_close, p.return_5d, p.return_20d, p.volatility_20d,
                 a.ai_sentiment, a.news_count, g.gdelt_tone, g.gdelt_events,
                 t.rsi_14, t.macd_hist, t.gap_pct, s.sector_momentum_5d, s.sector_breadth,
                 b.advance_ratio, b.pct_above_ma20, p.as_of_date
    """)
    features.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.ml_prediction_features")
    cnt = features.count()
    latest = spark.sql(f"SELECT MAX(pred_date) FROM {CATALOG}.gold.ml_prediction_features").first()[0]
    log_step('assemble_features', f'{CATALOG}.gold.ml_prediction_features', cnt)
    print(f'\u2705 gold.ml_prediction_features: {cnt} symbols, latest = {latest}')
except Exception as e:
    log_step('assemble_features', f'{CATALOG}.gold.ml_prediction_features', error=e)
    print(f'\u274c gold.ml_prediction_features: {e}')
    raise

In [0]:
# ── Step 2: Rebuild silver.ml_training_features with ground truth ─────────
# Historical feature vectors + actual_up label for last 30 days of trading
# Used by the training pipeline to retrain the ensemble model
try:
    training = spark.sql(f"""
        WITH dates AS (
            SELECT DISTINCT DATE(date) AS trade_date
            FROM {CATALOG}.silver.stock_prices
            WHERE DATE(date) >= DATE_SUB(CURRENT_DATE(), 30)
              AND DATE(date) < CURRENT_DATE()
              AND symbol IN ({sym_list})
        ),
        prices_hist AS (
            SELECT symbol, DATE(date) AS date, close,
                   (close - LAG(close, 5) OVER (PARTITION BY symbol ORDER BY date))
                    / NULLIF(LAG(close, 5) OVER (PARTITION BY symbol ORDER BY date), 0) AS return_5d,
                   (close - LAG(close, 20) OVER (PARTITION BY symbol ORDER BY date))
                    / NULLIF(LAG(close, 20) OVER (PARTITION BY symbol ORDER BY date), 0) AS return_20d,
                   STDDEV(close) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
                    / NULLIF(AVG(close) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW), 0) AS volatility_20d,
                   LEAD(close, 1) OVER (PARTITION BY symbol ORDER BY date) AS next_close
            FROM {CATALOG}.silver.stock_prices
            WHERE symbol IN ({sym_list})
              AND DATE(date) >= DATE_SUB(CURRENT_DATE(), 60)
        ),
        labeled AS (
            SELECT symbol, date AS pred_date,
                   DATE_SUB(date, 1) AS actual_date,
                   close AS last_close, return_5d, return_20d, volatility_20d,
                   CASE WHEN next_close > close THEN 1 ELSE 0 END AS actual_up,
                   (next_close - close) / NULLIF(close, 0) AS actual_return
            FROM prices_hist
            WHERE date IN (SELECT trade_date FROM dates)
              AND next_close IS NOT NULL
        ),
        tech AS (
            SELECT symbol, date, rsi_14, macd_hist, gap_pct
            FROM {CATALOG}.silver.technical_indicators
            WHERE date >= DATE_SUB(CURRENT_DATE(), 30)
        ),
        sector AS (
            SELECT symbol, date, sector_momentum_5d, sector_breadth
            FROM {CATALOG}.silver.sector_features
            WHERE date >= DATE_SUB(CURRENT_DATE(), 30)
        ),
        breadth AS (
            SELECT date, advance_ratio, pct_above_ma20
            FROM {CATALOG}.silver.market_breadth
            WHERE date >= DATE_SUB(CURRENT_DATE(), 30)
        ),
        ai AS (SELECT symbol, ai_sentiment, news_count FROM {CATALOG}.silver.news_ai_sentiment),
        gdelt_daily AS (
            SELECT symbol, event_date AS date, AVG(avg_tone) AS gdelt_tone, COUNT(*) AS gdelt_events
            FROM {CATALOG}.bronze.historical_news_gdelt
            WHERE event_date >= DATE_SUB(CURRENT_DATE(), 35) AND symbol IN ({sym_list})
            GROUP BY symbol, event_date
        ),
        macro_daily AS (
            SELECT date, 
                   MAX(CASE WHEN indicator='VIX' THEN value END) AS vix
            FROM {CATALOG}.bronze.fred_macro_indicators
            WHERE date >= DATE_SUB(CURRENT_DATE(), 35)
            GROUP BY date
        )
        SELECT l.symbol, l.pred_date, l.actual_date,
               l.return_5d, l.return_20d, l.volatility_20d,
               COALESCE(a.ai_sentiment, 0) AS ai_sentiment,
               COALESCE(a.news_count, 0) AS news_count,
               CAST(0 AS BIGINT) AS pos_articles,
               CAST(0 AS BIGINT) AS neg_articles,
               COALESCE(g.gdelt_tone, 0) AS gdelt_tone,
               COALESCE(CAST(g.gdelt_events AS BIGINT), CAST(0 AS BIGINT)) AS gdelt_events,
               COALESCE(t.rsi_14, 50) AS rsi_14,
               COALESCE(t.macd_hist, 0) AS macd_hist,
               0.0 AS bb_pct,
               1.0 AS vol_ratio,
               0.0 AS avg_range_5,
               COALESCE(t.gap_pct, 0) AS gap_pct,
               0.5 AS close_position,
               0.0 AS sector_rel_5d,
               COALESCE(s.sector_momentum_5d, 0) AS sector_momentum_5d,
               COALESCE(s.sector_breadth, 0.5) AS sector_breadth,
               0.0 AS stock_vs_sector_1d,
               0.0 AS market_return,
               COALESCE(b.advance_ratio, 0.5) AS advance_ratio,
               COALESCE(b.pct_above_ma20, 0.5) AS pct_above_ma20,
               0.0 AS market_dispersion,
               COALESCE(md.vix, 20) AS vix,
               0.0 AS hy_spread,
               0.0 AS treasury_10y,
               CAST(30 AS BIGINT) AS days_to_earnings,
               CAST(0 AS BIGINT) AS earnings_within_5d,
               CAST(DAYOFWEEK(l.pred_date) AS BIGINT) AS day_of_week,
               CAST(CASE WHEN DAYOFWEEK(l.pred_date) = 2 THEN 1 ELSE 0 END AS BIGINT) AS is_monday,
               l.actual_return,
               CAST(l.actual_up AS BIGINT) AS actual_up,
               CURRENT_TIMESTAMP() AS computed_at
        FROM labeled l
        LEFT JOIN ai a ON l.symbol = a.symbol
        LEFT JOIN gdelt_daily g ON l.symbol = g.symbol AND l.pred_date = g.date
        LEFT JOIN tech t ON l.symbol = t.symbol AND l.pred_date = t.date
        LEFT JOIN sector s ON l.symbol = s.symbol AND l.pred_date = s.date
        LEFT JOIN breadth b ON l.pred_date = b.date
        LEFT JOIN macro_daily md ON l.pred_date = md.date
    """)
    training.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.ml_training_features")
    cnt = spark.table(f"{CATALOG}.silver.ml_training_features").count()
    latest = spark.sql(f"SELECT MAX(pred_date) FROM {CATALOG}.silver.ml_training_features").first()[0]
    pct_up = spark.sql(f"SELECT AVG(actual_up) FROM {CATALOG}.silver.ml_training_features").first()[0]
    log_step('build_training', f'{CATALOG}.silver.ml_training_features', cnt)
    print(f'\u2705 silver.ml_training_features: {cnt} rows, latest = {latest}, {pct_up:.0%} went up')
except Exception as e:
    log_step('build_training', f'{CATALOG}.silver.ml_training_features', error=e)
    print(f'\u274c silver.ml_training_features: {e}')
    raise

In [0]:
# ── Step 3: Train ensemble inline + generate predictions ───────────────
# Uses same hyperparams as the registered model (LGB+RF+GB)
try:
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    import lightgbm as lgb
    import warnings
    warnings.filterwarnings('ignore')

    # Load training data
    train_df = spark.table(f"{CATALOG}.silver.ml_training_features").toPandas()
    y_train = train_df['actual_up'].values
    for f in CURATED_FEATURES:
        if f not in train_df.columns:
            train_df[f] = 0
    X_train = train_df[CURATED_FEATURES].fillna(0).values
    print(f"Training on {len(X_train)} samples, {len(CURATED_FEATURES)} features...")

    # Train 3 models with same hyperparams as registered model
    lgb_model = lgb.LGBMClassifier(num_leaves=8, learning_rate=0.1, n_estimators=50,
                                    min_child_samples=3, random_state=42, verbose=-1)
    lgb_model.fit(X_train, y_train)

    rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=3,
                                       random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    gb_model = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate=0.1,
                                           min_samples_leaf=3, random_state=42)
    gb_model.fit(X_train, y_train)
    print("\u2705 Ensemble trained (LGB + RF + GB)")

    # Predict on today's features
    features_pdf = spark.table(f"{CATALOG}.gold.ml_prediction_features").toPandas()
    for f in CURATED_FEATURES:
        if f not in features_pdf.columns:
            features_pdf[f] = 0
    X_pred = features_pdf[CURATED_FEATURES].fillna(0).values

    lgb_prob = lgb_model.predict_proba(X_pred)[:, 1]
    rf_prob = rf_model.predict_proba(X_pred)[:, 1]
    gb_prob = gb_model.predict_proba(X_pred)[:, 1]
    ensemble_prob = (lgb_prob + rf_prob + gb_prob) / 3

    features_pdf['direction'] = np.where(ensemble_prob > 0.5, 'UP', 'DOWN')
    features_pdf['probability_up'] = np.round(ensemble_prob, 4)
    features_pdf['confidence'] = np.round(np.abs(ensemble_prob - 0.5) * 2, 4)
    features_pdf['lgb_prob'] = np.round(lgb_prob, 4)
    features_pdf['rf_prob'] = np.round(rf_prob, 4)
    features_pdf['gb_prob'] = np.round(gb_prob, 4)

    pred_sdf = spark.createDataFrame(features_pdf)
    pred_sdf = pred_sdf.withColumn("computed_at", F.current_timestamp())
    pred_sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.ml_stock_predictions")

    cnt = spark.table(f"{CATALOG}.gold.ml_stock_predictions").count()
    latest = spark.sql(f"SELECT MAX(pred_date) FROM {CATALOG}.gold.ml_stock_predictions").first()[0]
    n_up = int((features_pdf['direction'] == 'UP').sum())
    n_down = int((features_pdf['direction'] == 'DOWN').sum())
    hi_conf = int((features_pdf['confidence'] > 0.4).sum())
    log_step('predict', f'{CATALOG}.gold.ml_stock_predictions', cnt)
    print(f'\u2705 gold.ml_stock_predictions: {cnt} stocks, latest = {latest}')
    print(f'   {n_up} UP / {n_down} DOWN | {hi_conf} high-confidence (>40%)')
except Exception as e:
    log_step('predict', f'{CATALOG}.gold.ml_stock_predictions', error=e)
    print(f'\u274c gold.ml_stock_predictions: {e}')
    raise

In [0]:
# ── Summary ─────────────────────────────────────────────────────────
print('=' * 60)
print('\U0001f9e0 ML PREDICTIONS REFRESH COMPLETE')
print('=' * 60)

ml_tables = [
    ('gold.ml_prediction_features', 'pred_date'),
    ('silver.ml_training_features', 'pred_date'),
    ('gold.ml_stock_predictions', 'pred_date'),
]

for table, date_col in ml_tables:
    try:
        r = spark.sql(f"SELECT MAX({date_col}) as latest, COUNT(*) as cnt FROM {CATALOG}.{table}").first()
        log_step('summary', f'{CATALOG}.{table}', r.cnt)
        print(f'  \u2705 {table}: {r.cnt:,} rows, latest = {r.latest}')
    except Exception as e:
        print(f'  \u274c {table}: {e}')

print('=' * 60)
logger.info('ML Predictions refresh completed successfully')